In [2]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

con = duckdb.connect("../data/processed/chess.duckdb")

games = con.sql("""
SELECT *
FROM games_matched
""").df()

print(f"Games: {len(games):,}")
games.head()

Games: 483,962


,event,site,date_raw,round,white,black,white_elo,black_elo,white_fide_id,black_fide_id,...,event_date,ply_count,result,eco,date,date_approx,result_score,game_id,white_id,black_id
0,SuperUnited CRO Rapid 2025,Zagreb CRO,2025.07.02,1.1,"Giri,A","Abdusattorov,Nodirbek",2748.0,2767.0,24116068,14204118,...,2025.07.02,<NA>,1-0,C70,2025-07-02,False,2,9e9749decef22e6005793728b7e4dab7,24116068,14204118
1,SuperUnited CRO Rapid 2025,Zagreb CRO,2025.07.02,1.2,"So,W","Caruana,F",2745.0,2777.0,5202213,2020009,...,2025.07.02,<NA>,1-0,C55,2025-07-02,False,2,56e32b781b2d746fe0ebdb5ae72690bd,5202213,2020009
2,SuperUnited CRO Rapid 2025,Zagreb CRO,2025.07.02,1.3,"Saric,Iv","Carlsen,M",2661.0,2837.0,14508150,1503014,...,2025.07.02,<NA>,1/2-1/2,B33,2025-07-02,False,1,6543bd7e8938587ffd0291da5a5d1971,14508150,1503014
3,SuperUnited CRO Rapid 2025,Zagreb CRO,2025.07.02,1.4,"Duda,J","Gukesh,D",2725.0,2776.0,1170546,46616543,...,2025.07.02,<NA>,1-0,C22,2025-07-02,False,2,8313264ce844414fac6c7ec4299b2ec4,1170546,46616543
4,SuperUnited CRO Rapid 2025,Zagreb CRO,2025.07.02,1.5,"Firouzja,Alireza","Praggnanandhaa,R",2766.0,2767.0,12573981,25059530,...,2025.07.02,<NA>,1/2-1/2,D20,2025-07-02,False,1,73e9fef600d7176b7b234264a6b8c66d,12573981,25059530


In [3]:
games["year"] = games["date"].dt.year

games["avg_elo"] = (
    games["white_elo"]
    + games["black_elo"]
) / 2

games[
    ["year", "avg_elo"]
].head()

,year,avg_elo
0,2025.0,2757.5
1,2025.0,2761.0
2,2025.0,2749.0
3,2025.0,2750.5
4,2025.0,2766.5


In [4]:
games_per_year = (
    games
    .groupby("year")
    .size()
    .reset_index(name="games")
)

games_per_year

,year,games
0,2023.0,539
1,2024.0,102
2,2025.0,185854
3,2026.0,297447


In [5]:
fig = px.bar(
    games_per_year,
    x="year",
    y="games",
    title="Games per Year"
)

fig.show()

In [6]:
result_counts = (
    games["result"]
    .value_counts()
    .reset_index()
)

result_counts.columns = [
    "result",
    "count"
]

result_counts

,result,count
0,1-0,210101
1,0-1,177439
2,1/2-1/2,96422


In [7]:
fig = px.pie(
    result_counts,
    names="result",
    values="count",
    title="Result Distribution"
)

fig.show()

In [8]:
total_games = len(games)

white_wins = (
    games["result"] == "1-0"
).sum()

draws = (
    games["result"] == "1/2-1/2"
).sum()

white_score = (
    white_wins +
    0.5 * draws
) / total_games

print(
    f"White Score = {white_score:.4f}"
)

print(
    f"White Advantage = {(white_score - 0.5)*100:.2f}%"
)

White Score = 0.5337
White Advantage = 3.37%


In [10]:
games["elo_band"] = pd.cut(
    games["avg_elo"],
    bins=[
        0,
        2200,
        2400,
        2600,
        2800,
        4000
    ],
    labels=[
        "<2200",
        "2200-2399",
        "2400-2599",
        "2600-2799",
        "2800+"
    ]
)

In [11]:
draw_rate = (
    games
    .groupby("elo_band")["result"]
    .apply(
        lambda s:
        (s == "1/2-1/2").mean() * 100
    )
    .reset_index(name="draw_rate")
)

draw_rate

,elo_band,draw_rate
0,<2200,17.011993
1,2200-2399,19.446091
2,2400-2599,25.260306
3,2600-2799,30.382335
4,2800+,51.920694


In [12]:
fig = px.bar(
    draw_rate,
    x="elo_band",
    y="draw_rate",
    title="Draw Rate by Elo Band",
    text="draw_rate"
)

fig.show()

In [13]:
elo_df = pd.concat([
    games["white_elo"],
    games["black_elo"]
])

elo_df = elo_df.dropna()

fig = px.histogram(
    elo_df,
    nbins=50,
    title="Elo Distribution"
)

fig.show()

In [14]:
white_score_by_elo = (
    games
    .groupby("elo_band")["result"]
    .apply(
        lambda s:
        (
            (s == "1-0").sum()
            +
            0.5 * (s == "1/2-1/2").sum()
        )
        / len(s)
    )
    .reset_index(name="white_score")
)

white_score_by_elo

,elo_band,white_score
0,<2200,0.530258
1,2200-2399,0.534814
2,2400-2599,0.537798
3,2600-2799,0.540022
4,2800+,0.664808


In [15]:
fig = px.bar(
    white_score_by_elo,
    x="elo_band",
    y="white_score",
    title="White Score by Elo Band",
    text="white_score"
)

fig.show()

In [20]:
top_openings = (
    games["eco"]
    .value_counts()
    .head(20)
    .reset_index()
)

top_openings.columns = [
    "eco",
    "games"
]

top_openings


,eco,games
0,A45,13191
1,A05,10236
2,D02,9775
3,B40,9210
4,B12,8612
5,B90,7754
6,A04,7482
7,A07,7437
8,B30,7116
9,B23,6893


In [17]:
fig = px.bar(
    top_openings,
    x="eco",
    y="games",
    title="Top 20 ECO Codes"
)

fig.show()


In [18]:
fig = px.bar(
    top_openings,
    x="eco",
    y="games",
    title="Top 20 ECO Codes"
)

fig.show()


In [21]:
eco_names = {
    "A45": "Trompowsky Attack",
    "D02": "Queen's Pawn Game",
    "B12": "Caro-Kann Defense",
    "A04": "Reti Opening",
    "B30": "Sicilian Defense",
    "D35": "Queen's Gambit Declined",
    "A00": "Uncommon Opening",
    "B06": "Modern Defense",
    "B22": "Sicilian Alapin",
    "C50": "Italian Game"
}

top_openings["opening"] = (
    top_openings["eco"]
    .map(eco_names)
)

top_openings

,eco,games,opening
0,A45,13191,Trompowsky Attack
1,A05,10236,NaN
2,D02,9775,Queen's Pawn Game
3,B40,9210,NaN
4,B12,8612,Caro-Kann Defense
5,B90,7754,NaN
6,A04,7482,Reti Opening
7,A07,7437,NaN
8,B30,7116,Sicilian Defense
9,B23,6893,NaN


In [28]:
# White score by ECO opening

opening_stats = (
    games
    .groupby("eco")
    .agg(
        games=("game_id", "count"),

        avg_elo=("avg_elo", "mean"),

        white_score=(
            "result",
            lambda s:
            (
                (s == "1-0").sum()
                +
                0.5 * (s == "1/2-1/2").sum()
            )
            / len(s)
        )
    )
    .reset_index()
)

# Keep only ECO codes with enough games
opening_stats = opening_stats[
    opening_stats["games"] >= 1000
]

# Round values
opening_stats["white_score"] = (
    opening_stats["white_score"]
    .round(3)
)

opening_stats["avg_elo"] = (
    opening_stats["avg_elo"]
    .round(0)
)

# Sort by white score
opening_stats = opening_stats.sort_values(
    "white_score",
    ascending=False
)

# Display top 20
top20_openings = opening_stats.head(20)

display(top20_openings)

# Plot
fig = px.bar(
    top20_openings,
    x="eco",
    y="white_score",
    color="games",
    hover_data=["avg_elo"],
    title="Top 20 Openings by White Score",
    text="white_score"
)

fig.update_layout(
    xaxis_title="ECO Code",
    yaxis_title="White Score"
)

fig.show()

,eco,games,avg_elo,white_score
378,D78,1094,2254.0,0.603
312,D11,1902,2220.0,0.600
132,B31,2909,2229.0,0.598
316,D15,1066,2242.0,0.598
31,A30,1104,2253.0,0.597
266,C65,1259,2366.0,0.596
12,A11,1076,2240.0,0.593
15,A14,2688,2297.0,0.584
1,A00,6552,2239.0,0.583
285,C84,1048,2370.0,0.583


In [29]:
white_score_by_elo = (
    games
    .groupby("elo_band")
    .apply(
        lambda g: (
            (g["result"] == "1-0").sum()
            +
            0.5 * (g["result"] == "1/2-1/2").sum()
        ) / len(g)
    )
    .reset_index(name="white_score")
)

white_score_by_elo

,elo_band,white_score
0,<2200,0.530258
1,2200-2399,0.534814
2,2400-2599,0.537798
3,2600-2799,0.540022
4,2800+,0.664808


In [30]:
fig = px.bar(
    white_score_by_elo,
    x="elo_band",
    y="white_score",
    title="White Score by Elo Band",
    text="white_score"
)

fig.update_traces(
    texttemplate="%{text:.3f}",
    textposition="outside"
)

fig.show()

In [31]:
top_players = (
    games
    .groupby("white")
    .agg(
        games=("game_id", "count"),
        avg_elo=("white_elo", "mean")
    )
    .query("games >= 20")
    .sort_values(
        "games",
        ascending=False
    )
    .head(20)
)

top_players

,games,avg_elo
white,,
"Rustemov,A",1208,2525.000000
"Terry,R",1088,2506.367647
"Sarana,A",990,2674.205051
"Bortnyk,Olexandr",975,2603.261538
"Maghsoodloo,Parham",817,2705.001224
"Vlassov,N",801,2317.459426
"Ambartsumova,K",794,2385.361461
"Lai,Duc Minh",784,2139.233418
"Bluebaum,M",771,2681.607004


In [32]:
top_events = (
    games
    .groupby("event")
    .size()
    .reset_index(name="games")
    .sort_values(
        "games",
        ascending=False
    )
    .head(20)
)

top_events

,event,games
485,46th Olympiad 2026,4513
486,46th Olympiad Women 2026,4132
214,26th ch-EUR Indiv 2026,2716
1716,Titled Tue 1st Jul Early,2561
1761,Titled Tue 8th Jul Early,2409
1851,World Blitz 2025,2384
1718,Titled Tue 1st Sep 2026,2335
1742,Titled Tue 29th Sep 2026,2314
1744,Titled Tue 2nd Jun 2026,2289
1705,Titled Tue 15th Sep 2026,2279


In [33]:
fig = px.bar(
    top_events,
    x="games",
    y="event",
    orientation="h",
    title="Top 20 Tournaments by Number of Games"
)

fig.show()